## Streaming processing of the QUAX experiment data with Dask and Kafka

## Overview

In this project, we build a **real-time ETL (Extract-Transform-Load) pipeline for streaming data** produced by the DAQ of the **QUAX experiment**, a resonant RF cavity used in axion dark matter searches. The DAQ produces scans consisting of the In-phase (I) and Quadrature (Q) components, which are two sinusoid functions in quadrature phase and together form a complex-valued time-domain signal (I + jQ). Processing this raw data consists of performing a **Fourier transform** on each scan to move from the time domain to the frequency domain, and then averaging all scans within a data-taking run to obtain a single averaged power spectrum. Each scan contains $2^{11} = 2048$ samples, and the FFT uses the same number of bins, while spanning a $2$ MHz bandwidth.

The dataset is composed of 2 sets of .dat binary files, one for the I measures and the other for the Q, each one comprised of a continuous series of ADC readings from the amplifier. Each ADC reading is written in the raw files as a 32-bit floating point value in little-endian format. Each file contains $2^{23}$ samples and its size is 32 MB. A realistic data production has a **throughput of 16 MB/s** and produces 1 file-pair every 5 seconds. 

## Architecture

The overall pipeline exploits **Dask** and **Apache Kafka** to perform the full real-time analysis. It consists of four main blocks:
- **Producer**: reads the raw I/Q `.dat` files and emulates a continuous DAQ stream, publishing batches of samples to Kafka with a configurable number of samples per batch and a configurable target throughput. It is hosted in a CloudVeneto virtual machine (VM).
- **Dask cluster**: consumes the raw batches from `topic_stream`, performs the FFT-based processing, and publishes the resulting power spectra to `topic_results`. It is deployed in CloudVeneto, and it hosts 1 scheduler node and up to 4 worker nodes.
- **Kafka broker**: acts as the decoupling layer between data management and processing, buffering the raw stream on `topic_stream` and the processed results on `topic_results`. It is hosted in the same CloudVeneto VM where the Dask scheduler sits.
- **Consumer**: subscribes to `topic_results` and renders a live-updating dashboard, showing both the per-worker spectra and the cumulative averaged spectrum. It is hosted on a local laptop.

Kafka sits between the Producer and the Dask cluster, and again between the Dask cluster and the Consumer, so that each stage can run independently and at its own pace: a slowdown in processing does not block data ingestion, and a slowdown in visualization does not block processing. This decoupling also makes the pipeline horizontally scalable, since additional Dask workers can be added to increase processing throughput without changing the Producer or Consumer.

We set up Kafka `topic_stream` with a partition number equals to or larger than the number of workers. Indeed, **each worker hosts a Kafka consumer and a Kafka producer**, and we bypass the data streaming through the Dask scheduler which can be a potential bottleneck. This set up also allows the creation of a consumer group of worker nodes that process data in parallel, favouring the construction of an easy scalable system.

The communications between our laptops and the VMs in CloudVeneto are hold via ssh. We ofter rely on port forwarding, which allows to connect our laptops (localhost) with CloudVeneto.

Each CloudVeneto VM has 2 vCPUs and 4GB of RAM (*flavor: cloudveneto-medium*).

![architecture.png](attachment:architecture.png)

### Kafka Broker Settings

The Kafka broker is hosted on the same CloudVeneto VM as the Dask scheduler. Below are the key configuration choices made to ensure the system operates smoothly, all of which are defined within the `kafka-server.properties` file used to start the Kafka broker:

* **Dual Network Listeners (`INTERNAL` / `EXTERNAL`)**: The broker exposes two separate communication channels on different ports. The `INTERNAL` listener (port 19092) is bound to the VM's private IP address (10.67.22.90) and handles all traffic within the CloudVeneto network, including communications with the Dask workers and the Producer. The `EXTERNAL` listener (port 9092) is advertised as `localhost`, allowing us to securely interact with the cluster from our personal laptops via an SSH tunnel.
* **Offset Replication Factor (`offsets.topic.replication.factor=1`)**: This parameter was set to 1, instead of the default value of 3. This adjustment is critical for a single-broker cluster: with the default value, Kafka would never be able to create the internal `__consumer_offsets` topic, which is essential for tracking data read states. Without this topic, Consumer Group coordination fails completely, causing consumers to hang indefinitely during the connection phase.
* **Data Retention Policies (`log.retention.*`)**: The log retention parameters were specifically tailored to the scale and duration of our tests. The total space limit was capped at approximately 192 MB (structured in 32 MB segments), while the time expiration was set to 30 minutes. Furthermore, we reduced the retention check frequency (`log.retention.check.interval.ms`) from the default 5 minutes to just 10 seconds. This responsiveness is necessary because the standard interval would not trigger the cleanup in time, leading to uncontrolled disk space growth and out-of-memory errors.
* **Large Payload Handling (`message.max.bytes`, `socket.*.bytes`)**: To prevent out-of-memory errors, the size limits for individual messages and network buffers (both receive and send) were raised to 100 MB. This capacity ensures the system to process the heaviest data batches, which can reach up to 64 MB in size plus the system headers.

```properties


# All settings are documented at https://kafka.apache.org/43/configuration/broker-configs/


# A comma-separated list of the names of the listeners used by the controller.
controller.listener.names=CONTROLLER

# The node id associated with this instance's roles
node.id=1

# The role of this server. Setting this puts us in KRaft mode
process.roles=broker,controller

# Listener name, hostname and port the broker or the controller will advertise to clients. INTERNAL refers to cloud VMs and EXTERNAL to our laptops
advertised.listeners=INTERNAL://10.67.22.90:19092,EXTERNAL://localhost:9092

# Disable the background thread checks the distribution of partition leaders at regular intervals. Not needed with a single broker
auto.leader.rebalance.enable=false

# List of controller endpoints used connect to the controller cluster
controller.quorum.bootstrap.servers=localhost:9093

# The address the socket server listens on
listeners=INTERNAL://0.0.0.0:19092,EXTERNAL://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093

# A comma separated list of directories under which to store log files
log.dirs=/tmp/kraft-combined-logs

# The frequency in milliseconds that the log retention check is triggered. We set this to 10 seconds
log.retention.check.interval.ms = 10000

# The minimum age of a log file to be eligible for deletion due to age
log.retention.minutes=30

# A size-based retention policy for logs. Functions independently of log.retention.hours. We set this to the size of 6 files (192MB)
log.retention.bytes=201326592

# The maximum size of a log segment file. When this size is reached a new log segment will be created. We set this to 32MB
log.segment.bytes=33554432

# The largest record batch size allowed by Kafka. We set this to 100MB
message.max.bytes=104857600

# The receive buffer (SO_RCVBUF) used by the socket server
socket.receive.buffer.bytes=104857600

# The maximum size of a request that the socket server will accept (protection against OOM)
socket.request.max.bytes=104857600

# The send buffer (SO_SNDBUF) used by the socket server
socket.send.buffer.bytes=1048576

# Name of listener used for communication between brokers.
inter.broker.listener.name=INTERNAL

# Maps listener names to security protocols.
listener.security.protocol.map=CONTROLLER:PLAINTEXT,INTERNAL:PLAINTEXT,EXTERNAL:PLAINTEXT

# The replication factor for the offsets topic.
offsets.topic.replication.factor=1

### Network Throughput Analysis

We are interested in a general **network throughput analysis** since our architecture completely relies on network communications. Specifically, the rate can be bottlenecked by the underlying network infrastructure. To evaluate this, we measured the throughput using `iperf3`, a bash command that evaluates the throughput between two communicating machines. 

Here, we compare the network throughput between a local laptop and a CloudVeneto VM against the internal throughput between two CloudVeneto nodes. This analysis was crucial for determining the optimal deployment location for the producer. We have found the following results:

*   **Throughput between a local laptop and a CloudVeneto VM:** When measuring through an SSH tunnel, the throughput ranges between 8 MB/s and 25 MB/s. This performance is highly dependent on the local Internet connection and exhibits significant variability even within the same network, making it difficult to guarantee a constant target throughput of 16 MB/s. We corroborated these findings using `speedtest-cli`, a command that measures the internet bandwidth, yielding upload speeds between 12 MB/s and 16 MB/s. While these latter measurements do not account for the SSH tunnel overhead, they provide a reliable baseline estimate.
*   **Throughput between CloudVeneto nodes:** The internal throughput consistently ranges between 375 MB/s and 500 MB/s, representing a highly stable and capable network link. In this context, using `speedtest-cli` is not meaningful, as it measures the connection outside the cluster. 

Based on these findings, **we deployed the producer directly within the CloudVeneto infrastructure** rather than on a local machine, successfully ensuring an ideal 16 MB/s target throughput. Moreover, note that this network measurement is an upper limit of the real throughput achieved by the Producer, which depends on other factors (see later).

Below, we report an excerpt of the terminal script with the tests performed on a local laptop connected via SSH to a CloudVeneto VM:

```bash
dghezzi@PcDaniele:~/Project_MAPD$ iperf3 -c localhost -p 15201
Connecting to host localhost, port 15201
[  5] local 127.0.0.1 port 44338 connected to 127.0.0.1 port 15201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.02   sec  26.1 MBytes   215 Mbits/sec    5   1.19 MBytes
[  5]   1.02-2.02   sec  18.4 MBytes   154 Mbits/sec    3   1.19 MBytes
[  5]   2.02-3.00   sec  19.9 MBytes   170 Mbits/sec    3   1.19 MBytes
[  5]   3.00-4.01   sec  19.8 MBytes   165 Mbits/sec    5   1.19 MBytes
[  5]   4.01-5.01   sec  21.2 MBytes   178 Mbits/sec    4   1.19 MBytes
[  5]   5.01-6.00   sec  18.2 MBytes   153 Mbits/sec    3   1.19 MBytes
[  5]   6.00-7.00   sec  20.2 MBytes   170 Mbits/sec    6   1.19 MBytes
[  5]   7.00-8.00   sec  19.6 MBytes   165 Mbits/sec    5   1.19 MBytes
[  5]   8.00-9.00   sec  18.2 MBytes   153 Mbits/sec    4   1.19 MBytes
[  5]   9.00-10.05  sec  22.1 MBytes   178 Mbits/sec    5   1.19 MBytes
- - - - - - - - - - - - - - - - - - - - - - - - -
[ ID] Interval           Transfer     Bitrate         Retr
[  5]   0.00-10.05  sec   205 MBytes   171 Mbits/sec   43             sender
[  5]   0.00-9.69   sec   199 MBytes   172 Mbits/sec                  receiver

dghezzi@PcDaniele:~/Project_MAPD$ iperf3 -c localhost -p 15201
Connecting to host localhost, port 15201
[  5] local 127.0.0.1 port 60206 connected to 127.0.0.1 port 15201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.01   sec  24.2 MBytes   202 Mbits/sec    3   3.68 MBytes
[  5]   1.01-2.00   sec  14.2 MBytes   120 Mbits/sec    1   3.68 MBytes
[  5]   2.00-3.01   sec  13.0 MBytes   109 Mbits/sec    0   3.68 MBytes
[  5]   3.01-4.00   sec  9.50 MBytes  80.1 Mbits/sec    3   3.68 MBytes
[  5]   4.00-5.00   sec  13.6 MBytes   114 Mbits/sec    0   3.68 MBytes
[  5]   5.00-6.01   sec  12.9 MBytes   108 Mbits/sec    3   3.68 MBytes
[  5]   6.01-7.00   sec  13.2 MBytes   112 Mbits/sec    1   3.68 MBytes
[  5]   7.00-8.00   sec  11.5 MBytes  96.4 Mbits/sec    0   3.68 MBytes
[  5]   8.00-9.01   sec  13.6 MBytes   114 Mbits/sec    2   3.68 MBytes
[  5]   9.01-10.01  sec  13.2 MBytes   111 Mbits/sec    3   3.68 MBytes
- - - - - - - - - - - - - - - - - - - - - - - - -
[ ID] Interval           Transfer     Bitrate         Retr
[  5]   0.00-10.01  sec   139 MBytes   117 Mbits/sec   16             sender
[  5]   0.00-9.76   sec   131 MBytes   113 Mbits/sec                  receiver

dghezzi@PcDaniele:~/Project_MAPD$ speedtest-cli
Retrieving speedtest.net configuration...
Testing from Telecom Italia Business (79.1.175.151)...
Retrieving speedtest.net server list...
Selecting best server based on ping...
Hosted by Wolnet Srl (Verona) [71.61 km]: 25.428 ms
Testing download speed...................................................
Download: 122.60 Mbit/s
Testing upload speed.....................................................
Upload: 109.69 Mbit/s

Here we report the connection tests between two distinct CloudVeneto VM:

```bash
debian@mapd-group11-4:~$ iperf3 -c 10.67.22.90
Connecting to host 10.67.22.90, port 5201
[  5] local 10.67.22.236 port 60000 connected to 10.67.22.90 port 5201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.00   sec   468 MBytes  3.92 Gbits/sec  1039   1.54 MBytes
[  5]   1.00-2.00   sec   479 MBytes  4.02 Gbits/sec    0   1.74 MBytes
[  5]   2.00-3.00   sec   388 MBytes  3.25 Gbits/sec    0   1.84 MBytes
[  5]   3.00-4.00   sec   467 MBytes  3.92 Gbits/sec   17   1.48 MBytes
[  5]   4.00-5.00   sec   442 MBytes  3.71 Gbits/sec    0   1.67 MBytes
[  5]   5.00-6.00   sec   372 MBytes  3.12 Gbits/sec    0   1.76 MBytes
[  5]   6.00-7.00   sec   358 MBytes  3.00 Gbits/sec    0   1.85 MBytes
[  5]   7.00-8.00   sec   348 MBytes  2.91 Gbits/sec    0   1.91 MBytes
[  5]   8.00-9.00   sec   389 MBytes  3.26 Gbits/sec    0   2.04 MBytes
[  5]   9.00-10.00  sec   441 MBytes  3.70 Gbits/sec   64   1.63 MBytes
- - - - - - - - - - - - - - - - - - - - - - - - -
[ ID] Interval           Transfer     Bitrate         Retr
[  5]   0.00-10.00  sec  4.05 GBytes  3.48 Gbits/sec  1120            sender
[  5]   0.00-10.00  sec  4.05 GBytes  3.48 Gbits/sec                  receiver

### Producer

The **Producer emulates the QUAX DAQ** by reading I/Q raw data and streaming it to Kafka as if it were being acquired live. At the start, it loads the full set of `duck_i_*.dat`/`duck_q_*.dat` file pairs into memory as **raw byte buffers**, avoiding repeated disk I/O during the streaming loop.

The stream is then played in fixed size batches, controlled by two configurable parameters:
- **`SCANS_PER_BATCH`**: the number of scans grouped into a single Kafka message.
- **`TARGET_THROUGHPUT`**: the target data rate (in bytes/s) the Producer tries to sustain, matching the real DAQ's rate of 16 MB/s.

For each batch, the **I and Q byte slices are concatenated into a single datum** and published to `topic_stream`: thus we can replicate exactly the DAQ, which sends a pair of I and Q files every 5 seconds, by setting `SCANS_PER_BATCH` $=4096$. Each message contains also a set of **headers** (producer timestamp, target throughput, and scans per batch) used downstream for latency benchmarking. Then the **Producer calls `send()` and `flush()`**: the first function append in a memory buffer the message and returns a future object, while the second function pushes all the buffered messages to the broker (and wait for their acknowledgment). This logic allows to implement a control over the pacing of each batch, necessary to reliably fix `SCANS_PER_BATCH` and `TARGET_THROUGHPUT` as independent variables rather than letting Kafka's internal batching logic determine them implicitly.

The Producer is run from a dedicated VM inside CloudVeneto (connecting directly to the `INTERNAL` listener).

In [ ]:
from kafka import KafkaProducer
import glob
import sys
import time
import os
import datetime

print("Initializing kafka producer")

# =============================================================================
# Configuration
# =============================================================================
KAFKA_TOPIC = "topic_stream"
BOOTSTRAP_SERVERS = "localhost:9092"

'''
# to see Kafka debug logs
import logging
logging.basicConfig(level=logging.DEBUG)
logging.getLogger("kafka").setLevel(logging.DEBUG)
'''

# =============================================================================
# Kafka Producer Setup
# =============================================================================
# All settings explained in https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    #client_id                  # Appears in server-side logs. Default: ‘kafka-python-producer-#’ (appended with a unique number per instance)
    key_serializer = None,      # Send raw bytes
    value_serializer = None,    # Send raw bytes
    #transactional_id           # Not needed
    enable_idempotence = True,  # Ensure that exactly one copy of each message is written in the stream (Commented out for kafka-python-ng compatibility)
    #delivery_timeout_ms        # Not needed
    #acks = 0,                  # Use default all
    compression_type=None,      # No compression
    #retries                    # Not needed
    batch_size = 0,             # Disable batching, we will do our own
    #linger_ms                  # Not needed, batching is disabled
    #partitioner                # Use default
    #connections_max_idle_ms    # Use default
    max_block_ms = 3000,                        # Max block time if buffer is full
    max_request_size = 68*1024*1024,           # The maximum size of a request. This is also effectively a cap on the maximum record size. Note that the server has its own cap on record size which may be different from this
    max_in_flight_requests_per_connection = 1, # Requests are pipelined to kafka brokers up to this number of maximum requests per broker connection
    security_protocol = "PLAINTEXT",    
)

producer.partitions_for(KAFKA_TOPIC) # Warm up the producer to fetch cluster metadata

# =============================================================================
# Data Loading
# =============================================================================
# Locate and pair the I and Q data files
i_files = sorted(glob.glob("data/duck_i_*.dat"))
q_files = sorted(glob.glob("data/duck_q_*.dat"))
assert len(i_files) == len(q_files) == 31
files_count = len(i_files)

# Ensure file size matches expectations
FILE_SIZE = os.path.getsize(i_files[0])
assert FILE_SIZE == 33554432
HALF_TOTAL_SIZE = FILE_SIZE * files_count

print("Loading files...\n")

# Pre-allocate large bytearrays to hold all I and Q data in memory
i_data = bytearray(HALF_TOTAL_SIZE)
q_data = bytearray(HALF_TOTAL_SIZE)

# Read files sequentially directly into the pre-allocated buffers
for idx, (i_file, q_file) in enumerate(zip(i_files, q_files)):
    sys.stdout.write(f"\rLoading files {idx + 1}/{len(i_files)}")
    sys.stdout.flush()

    start = idx * FILE_SIZE
    end = start + FILE_SIZE

    # Read I data
    with open(i_file, "rb") as f:
        n = f.readinto(memoryview(i_data)[start:end])
        assert n == FILE_SIZE

    # Read Q data
    with open(q_file, "rb") as f:
        n = f.readinto(memoryview(q_data)[start:end])
        assert n == FILE_SIZE
print("\n")

# =============================================================================
# Streaming Parameters
# =============================================================================
SCANS_PER_BATCH = 4096
TARGET_THROUGHPUT = 16*1024*1024 # Target throughput in Bytes/s (16384 = 16 kB/s, final target will be 16777216 for 16 MB/s)

SAMPLES_PER_SCAN = 2048
SAMPLES_PER_BATCH = SCANS_PER_BATCH * SAMPLES_PER_SCAN
HALF_BATCH_SIZE = 4 * SAMPLES_PER_BATCH # 4 bytes per float32 sample

# Time to wait between batch transmissions to maintain TARGET_THROUGHPUT
TARGET_PUBLISH_INTERVAL = 2 * HALF_BATCH_SIZE / TARGET_THROUGHPUT

n_batches = HALF_TOTAL_SIZE // HALF_BATCH_SIZE

print(f"Streaming {n_batches} batches of size {2*HALF_BATCH_SIZE}B...")

# =============================================================================
# Main Streaming Loop
# =============================================================================
try:
    start_time = time.perf_counter()
    
    for i in range(n_batches):
        # Calculate slicing indices for the current batch
        start = i * HALF_BATCH_SIZE
        end = start + HALF_BATCH_SIZE

        # Extract I and Q bytes for this batch
        i_bytes = i_data[start:end]
        q_bytes = q_data[start:end]

        packet = i_bytes + q_bytes # Concatenate I and Q into a single packet
        
        timestamp = time.time()
        print(f"[{datetime.datetime.fromtimestamp(timestamp)}] Sending batch {i+1}/{n_batches} (scans up to {(i+1)*SCANS_PER_BATCH})")

        # Send packet to Kafka and measure the time for sending
        future = producer.send(
            KAFKA_TOPIC, 
            value=packet, 
            headers=[('producer_ts', str(timestamp).encode('utf-8')), ('throughput', str(TARGET_THROUGHPUT).encode('utf-8'))] # Attach metadata
        )
        producer.flush()

        # Calculate the target time and sleep if necessary to maintain the target publish interval
        target_time = start_time + (i + 1) * TARGET_PUBLISH_INTERVAL
        sleep_time = target_time - time.perf_counter()
        
        if sleep_time > 0:
            time.sleep(sleep_time)
        else:
            # If processing and network I/O took longer than the interval, warn about falling behind
            print(f"missed send by {sleep_time}s")
            
finally:
    print("Closing producer...")
    producer.close()

print("\nDone streaming.")

### Dask Cluster 

The **Dask cluster** is deployed in CloudVeneto and consists of one scheduler node and up to 4 worker nodes, each running on a VM with 2 vCPUs and 4GB of RAM. The scheduler is located on the same VM as the Kafka broker. Unlike a typical Dask workload, the scheduler only launches one long-running task per worker through `dask_client.submit`. Beyond launching these tasks, the scheduler is not involved in the data path at all: **each worker owns its own Kafka consumer and producer, so the raw stream and the results stream both bypass the scheduler entirely and flow directly between the workers and the broker**. This avoids turning the scheduler into a throughput bottleneck and keeps the design close to a Kafka consumer-group pattern rather than a classic Dask task graph.

Each worker's consumer joins the same **consumer group**, allowing parallel processing of partitioned data. For every batch pulled from `topic_stream`, the worker reshapes the I/Q samples into a `(n_scans, SAMPLES_PER_SCAN)` matrix, allowing vectorized single-batch processing: it computes a row-wise FFT to obtain the power spectrum of each scan, then evaluates the mean and second moment across all scans. Rather than storing every scan, the worker maintains a running mean and a running sum of squared deviations of the power spectrum, updated using [Welford's online algorithm](https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Welford's_online_algorithm): this saves memory and keeps its usage constant regardless of how long the stream runs.

Since we aim to refresh the Consumer dashboard every 5 seconds, we set `publish_interval_sec` accordingly: once this interval has elapsed, the worker publishes the accumulated mean, M2 and scan count together with batch metadata (producer timestamps, processing times, queueing times) to `topic_results`, then resets its accumulators for the next time interval. On receiving the producer's EOF marker, each worker flushes any remaining partial accumulation and forwards an EOF message downstream, so the Consumer can detect when all workers have finished.

Here the code where we define the Dask SSH Cluster and worker task:

In [ ]:
from dask.distributed import Client, SSHCluster

print("Creating Dask SSH cluster...")
dask_cluster = SSHCluster(
    [SCHEDULER_IP] + WORKER_IPS,
    connect_options = {"known_hosts": None}, # disable server host key validation
    worker_options = {"n_workers": 1, "nthreads": 1}, # Keywords to pass on to workers
    scheduler_options={"port": 8786, "dashboard_address": "0.0.0.0:8797"} # Keywords to pass on to scheduler
)

dask_client = Client(dask_cluster, name="dask_client")
dask_client

In [ ]:
def process_batch(bytestring):
    import numpy as np
    
    SAMPLES_PER_SCAN = 2048 # Number of complex samples in each scan
    BYTES_PER_SCAN = 8 * SAMPLES_PER_SCAN
    SAMPLING_FREQ_HZ = 2e6 # ADC readout frequency

    batch_size = len(bytestring)
    assert batch_size % BYTES_PER_SCAN == 0
    n_scans = batch_size // BYTES_PER_SCAN

    # Split raw bytes into I and Q float32 arrays
    i_data = np.frombuffer(bytestring[:batch_size//2], dtype="<f4") # < = little-endian, f4 = float32
    q_data = np.frombuffer(bytestring[batch_size//2:], dtype="<f4")
    complex_signal = np.empty(i_data.shape, dtype=np.complex64) # this way avoids intermediate allocation
    complex_signal.real = i_data
    complex_signal.imag = q_data

    # Shape the signal into a matrix to perform vectorized FFT row-wise
    matrix = complex_signal.reshape((n_scans, SAMPLES_PER_SCAN))
    # Compute centered frequency spectrum
    spectra = np.fft.fft(matrix, axis=1)
    power_spectra = np.abs(spectra) ** 2

    # Compute mean and second moment of power across scans
    power_means = np.mean(power_spectra, axis=0)
    power_M2s = np.sum((power_spectra - power_means) ** 2, axis=0)

    # Compute frequency bins
    frequencies = np.fft.fftfreq(SAMPLES_PER_SCAN, d = 1 / SAMPLING_FREQ_HZ) # could be optimized out of the function

    return {
        "n_scans": n_scans,
        "frequencies": frequencies,
        "means": power_means,
        "M2s": power_M2s
    }

def worker_function(worker_id, bootstrap_servers, input_topic, output_topic, publish_interval_sec):
    # Worker imports are inside the function so they execute on the worker process
    import json
    import numpy as np
    import time
    from kafka import KafkaConsumer, KafkaProducer

    # Setup consumer. This will read data from the DAQ
    consumer = KafkaConsumer(
        input_topic,
        bootstrap_servers=bootstrap_servers,
        client_id = f"worker{worker_id}_consumer", # Appears in server-side logs
        group_id = 'dask_processing_group_v3',        # consumer group to join for dynamic partition assignment
        group_instance_id = f"worker{worker_id}_consumer_instance", # Make this consumer a static member of the group (not really necessary)
        value_deserializer = None,                 # Deserialization will be performed manually
        max_partition_fetch_bytes = 32*1024*1024,  # This size must be at least as large as the maximum message size the server allows
        auto_offset_reset='earliest',              # Bring the reading offset to the earliest message
        security_protocol = "PLAINTEXT", 
    )
    
    # Setup producer. This will publish data for the dashboard
    producer = KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        client_id = f"worker{worker_id}_producer", # Appears in server-side logs
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        compression_type=None,                     # No compression
        enable_idempotence = True,                 # Ensure that exactly one copy of each message is written in the stream
        batch_size = 0,                            # Disable batching, we will do our own
        max_block_ms = 3000,                       # Max block time if buffer is full
        max_request_size = 256*1024,               # The maximum size of a request. This is also effectively a cap on the maximum record size. Note that the server has its own cap on record size which may be different from this
        max_in_flight_requests_per_connection = 1, # Requests are pipelined to kafka brokers up to this number of maximum requests per broker connection
        security_protocol = "PLAINTEXT"
    )

    # Setup accumulators and publish deadline
    accumulated_means = np.array([])
    accumulated_M2s = np.array([])
    accumulated_n_scans = 0
    frequencies = np.array([])
    producer_timestamps = []
    batch_completion_times = []
    processing_times = []
    publish_deadline = time.monotonic() + publish_interval_sec
    producer_throughput = -1.0

    eof = False
    try:
        for msg in consumer:
            # Read metadata (msg header) provided by the producer
            if msg.headers:
                for key, value in msg.headers:
                    if key == "producer_ts":
                        producer_timestamp = float(value.decode('utf-8'))
                    elif key == "throughput":
                        producer_throughput = float(value.decode('utf-8'))
                    elif key == "EOF":
                        if accumulated_n_scans != 0:
                            result = {
                                "worker_id": worker_id,
                                "batches_details": {
                                    "producer_timestamps": producer_timestamps,                           # Timestamp reported by producer for each batch
                                    "waiting_times": [now - finish_time for finish_time in batch_completion_times], # How long each batch has been held due to publishing interval logic
                                    "processing_times" : processing_times,                                # Time to process each batch
                                    "throughput": producer_throughput,
                                    "scans_per_batch": batch_n_scans
                                },
                                "results": {
                                    "n_averaged_scans": accumulated_n_scans,
                                    "frequencies": frequencies.tolist(),
                                    "power_means": accumulated_means.tolist(),
                                    "power_M2s": accumulated_M2s.tolist()
                                } 
                            }
                            producer.send(output_topic, value=result)
                            producer.flush()
                        producer.send(output_topic, value="{}", headers=[('EOF', b"")])
                        producer.flush()
                        eof = True
            if eof:
                break
                        
            producer_timestamps.append(producer_timestamp)
    
            # Process the batch
            processing_time_start = time.perf_counter()
            
            batch_result = process_batch(msg.value)
            batch_n_scans = batch_result["n_scans"]
            batch_means = batch_result["means"]
            batch_M2s = batch_result["M2s"]
    
            # Update accumulators
            if accumulated_n_scans == 0:
                frequencies= batch_result["frequencies"]
                accumulated_means = batch_means.copy()
                accumulated_M2s = batch_M2s.copy()
                accumulated_n_scans = batch_n_scans
            else:
                deltas = batch_means - accumulated_means
                total_n_scans = accumulated_n_scans + batch_n_scans
                accumulated_means += deltas * batch_n_scans / total_n_scans
                accumulated_M2s += batch_M2s + deltas**2 * accumulated_n_scans * batch_n_scans / total_n_scans
                accumulated_n_scans = total_n_scans
            
            # Batch processing finished
            processing_time = time.perf_counter() - processing_time_start
            processing_times.append(processing_time)
    
            # Publish results if publish_deadline surpassed
            now = time.monotonic()
            batch_completion_times.append(now)
            if now > publish_deadline:
                print("publishing")

                # Publish result
                result = {
                    "worker_id": worker_id,
                    "batches_details": {
                         "producer_timestamps": producer_timestamps,                           # Timestamp reported by producer for each batch
                         "waiting_times": [now - finish_time for finish_time in batch_completion_times], # How long each batch has been held due to publishing interval logic
                        "processing_times" : processing_times,                                # Time to process each batch
                        "throughput": producer_throughput,
                         "scans_per_batch": batch_n_scans
                    },
                    "results": {
                         "n_averaged_scans": accumulated_n_scans,
                        "frequencies": frequencies.tolist(),
                        "power_means": accumulated_means.tolist(),
                        "power_M2s": accumulated_M2s.tolist()
                    } 
                }
                producer.send(output_topic, value=result)
                producer.flush()

                # Reset variables
                accumulated_means = np.array([])
                accumulated_M2s = np.array([])
                accumulated_n_scans = 0
                frequencies = np.array([])
                producer_timestamps = []
                batch_completion_times = []
                processing_times = []
                publish_deadline = time.monotonic() + publish_interval_sec
            else:
                print("skip publishing")
    finally:
        consumer.close()
        producer.close()

Finally, we submit the task to each worker node and we monitor their status:

In [ ]:
def report_status(future):
    if future.status == "error":
        print(f"Task {future.key} failed:")
        print(future.exception())
    else:
        print(f"Task {future.key} finished with status {future.status}")

for (i, worker_ip) in enumerate(WORKER_IPS):
    future = dask_client.submit(
        worker_function, # function to be executed
        # Parameters to function
        worker_id = i + 1,
        bootstrap_servers = BOOTSTRAP_SERVER,
        input_topic = TOPIC_STREAM,
        output_topic = TOPIC_RESULTS,
        publish_interval_sec = WORKER_PUBLISH_INTERVAL_SEC,
        # Dask parameters
        key = f"worker{i+1}", # Identifier for the task
        workers = worker_ip, # Restrict execution to the specific worker
        resources = {}, # no resources (eg GPUs) required
    )
    future.add_done_callback(report_status)


### Consumer

The **Consumer is a live matplotlib dashboard** that subscribes to `topic_results` and runs locally on our laptops. We set the connection with `auto_offset_reset="latest"` since the dashboard simply has to read and display whatever is newest. Messages are fetched via `consumer.poll(timeout_ms=100)` inside the main event loop, so that the dashboard keeps updating and stays responsive for incoming new messages. The Consumer also **saves the results useful for benchmarking in a JSON file**.

For each message, the Consumer updates a worker state and then merges that worker's partial spectrum into a single cumulative global spectrum using Welford's algorithm. This lets the dashboard show both the individual worker spectra and one combined average spectrum, with global plot error bars obtained from the accumulated second moment. The plot layout is rebuilt dynamically whenever the number of active workers changes, since the number of workers is not known in advance.

Alongside the plots, every incoming message is used to compute and log network latency (`receive_ts - producer_timestamp`) and FFT/processing latency, which are appended to `benchmark_records` and written out to a JSON file with other metadata, for later offline analysis. The exit condition itself is driven by EOF messages: each worker sends its own EOF marker once the Producer's stream ends, and the Consumer keeps a running count of the EOFs received, breaking out of the polling loop once it has seen one from every worker.

In [ ]:
import json
import time
from collections import deque
import matplotlib.pyplot as plt
import numpy as np
from kafka import KafkaConsumer

# =============================================================================
# Configuration
# =============================================================================
TOPIC = "topic_results"
BOOTSTRAP = "localhost:9092"

FREQ_MIN_HZ = -1.1e6
FREQ_MAX_HZ = 1.1e6
POWER_MIN = 1e-4
POWER_MAX = 30

BENCHMARK_FILE = "benchmark_try1.json"


# =============================================================================
# Kafka consumer
# =============================================================================
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    client_id="dashboard",     
    group_id=None,
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="latest", # Ignore old data, read only new incoming messages
    security_protocol="PLAINTEXT"
)

# =============================================================================
# Dashboard Setup
# =============================================================================
plt.ion() # Enable matplotlib interactive mode for real-time plotting
fig = plt.figure(figsize=(15, 7))

# Set up the main global plot (top half of the window)
ax_global = plt.subplot2grid((2, 1), (0, 0))
global_line, = ax_global.plot([], [], lw=2)
ax_global.set_xlabel("Frequency (Hz)")
ax_global.set_ylabel("Power")
ax_global.set_xlim(FREQ_MIN_HZ, FREQ_MAX_HZ)
ax_global.set_ylim(POWER_MIN, POWER_MAX)
ax_global.grid(True)

worker_states = {} # Stores data and logs for each worker
worker_plots = {}  # Stores matplotlib objects for each worker's subplot

# Variables to keep track of the cumulative spectrum across all workers
global_frequencies = None
global_power_means = None
global_power_M2s = None
global_n_averaged_scans = 0

# =============================================================================
# Main loop
# ============================================================================= 
print(f"Listening for messages: latencies will be saved to {BENCHMARK_FILE}")

n_eof = 0
plot_start_time = None

throughput = None
n_scans_per_batch = None

benchmark_records = []

class eof_exit(Exception):
    pass

try:
    # Keep running as long as the matplotlib window is open
    while plt.fignum_exists(fig.number):
        
        # Fetch new messages (max 100ms wait)
        records = consumer.poll(timeout_ms=100)

        for _, messages in records.items():
            for msg in messages:
                eof = False
                receive_ts = time.time()

                if msg.headers:
                    for key, value in msg.headers:
                        if key == "EOF":
                            print("EOF")
                            eof = True
                            n_eof += 1
                if n_eof != 0 and n_eof == n_workers:
                    print(f"Received {n_eof} EOFs, exiting")
                    raise eof_exit()
                if eof:
                     continue

                packet = msg.value
    
                worker_id = packet["worker_id"]
                
                # Register a new worker if seen for the first time
                if worker_id not in worker_states:
                    worker_states[worker_id] = {
                        "frequencies": None,
                        "power_means": None,
                        "power_M2s": None,
                        "n_averaged_scans": 0,

                        "producer_tss": None,
                        "receive_tss": [],
                        "waiting_times": None,
                        "processing_times": None,
                        
                        "log": deque(maxlen=8), # Keep only the last 8 log lines
                    }
                worker = worker_states[worker_id]

                # Extract results from the packet
                results = packet["results"]
                worker["n_averaged_scans"] = results["n_averaged_scans"]
                worker["frequencies"] = np.asarray(results["frequencies"])
                worker["power_means"] = np.asarray(results["power_means"])
                worker["power_M2s"] = np.asarray(results["power_M2s"])

                # Extract timing metrics
                metrics = packet["batches_details"]
                worker["producer_tss"] = metrics["producer_timestamps"]
                worker["waiting_times"] = metrics["waiting_times"]
                worker["processing_times"] = metrics["processing_times"]
                if n_scans_per_batch is None:
                    n_scans_per_batch = metrics["scans_per_batch"]
                else:
                    assert n_scans_per_batch == metrics["scans_per_batch"]
                if throughput is None:
                    throughput = metrics["throughput"]
                else:
                    assert throughput == metrics["throughput"]
                worker["receive_tss"].append(receive_ts)
            
                
                # Update global cumulative spectrum using a running average
                if global_power_means is None:
                    if plot_start_time is None:
                        plot_start_time = time.time()
                    global_frequencies = worker["frequencies"]
                    global_power_means = worker["power_means"].copy()
                    global_power_M2s = worker["power_M2s"].copy()
                    global_n_averaged_scans = worker["n_averaged_scans"]
                else:
                    # Compute the difference and update weights based on scans
                    delta = worker["power_means"] - global_power_means
                    total_scans = global_n_averaged_scans + worker["n_averaged_scans"]
                    global_power_means += delta * worker["n_averaged_scans"] / total_scans
                    global_power_M2s += worker["power_M2s"] + delta**2 * global_n_averaged_scans * worker["n_averaged_scans"] / total_scans
                    global_n_averaged_scans = total_scans

                # Parse latencies and save benchmarks
                producer_tss = metrics.get("producer_timestamps", [])
                processing_times_s = metrics.get("processing_times", [])
                

                net_latencies_ms = []
                for t_start in producer_tss:
                    if t_start is not None:
                        # Network latency = Total time (producer-consumer) - Time spent in the network and VMs
                        network_latency = (receive_ts - t_start) * 1000
                        net_latencies_ms.append(network_latency)

                fft_latencies = [t * 1000 for t in processing_times_s]

                if net_latencies_ms:
                    worker["last_net_mean"] = np.mean(net_latencies_ms)
                    worker["last_net_p95"] = np.percentile(net_latencies_ms, 95)
                else:
                    worker["last_net_mean"] = 0.0
                    worker["last_net_p95"] = 0.0

                if fft_latencies:
                    worker["last_fft_mean"] = np.mean(fft_latencies)
                    worker["last_fft_p95"] = np.percentile(fft_latencies, 95)
                else:
                    worker["last_fft_mean"] = 0.0
                    worker["last_fft_p95"] = 0.0

                # Save Benchmark
                benchmark_records.append({
                        "worker_id": worker_id,
                        "n_averaged_scans": worker["n_averaged_scans"],
                        "production_tss": worker["producer_tss"],
                        "receive_tss": worker["receive_tss"],
                        "net_latencies_ms": net_latencies_ms,
                        "fft_latencies_ms": fft_latencies,
                        "waiting_times": worker["waiting_times"]
                    })
                    

                # Print on the dashboard
                worker["log"].appendleft(
                    f"{time.strftime('%H:%M:%S')} | "
                    f"Net [Mean: {worker['last_net_mean']:.1f}ms, P95: {worker['last_net_p95']:.1f}ms] | "
                    f"FFT [Mean: {worker['last_fft_mean']:.1f}ms, P95: {worker['last_fft_p95']:.1f}ms]"
                )

                # Print on the console
                print(
                    f"{time.strftime('%H:%M:%S')} worker={worker_id} scans={worker['n_averaged_scans']} | "
                    f"Net (mean={worker['last_net_mean']:.1f}ms, p95={worker['last_net_p95']:.1f}ms) | "
                    f"FFT (mean={worker['last_fft_mean']:.1f}ms, p95={worker['last_fft_p95']:.1f}ms)"
                )

        # -------------------------------------------------------------------------
        # Interface rendering
        # -------------------------------------------------------------------------
        # Dynamically rebuild the figure if the number of workers changes
        if len(worker_plots) != len(worker_states):
            fig.clf()
            n_workers = max(len(worker_states), 1)
            # Create a layout: 1 top row for global, 1 middle row for spectra, 1 bottom row for text logs
            grid = fig.add_gridspec(3, n_workers, height_ratios=[2.0, 0.9, 1.2], hspace=0.35)

            ax_global = fig.add_subplot(grid[0, :])
            global_line, = ax_global.plot([], [], lw=2)
            global_error = ax_global.errorbar([], [], yerr=[], fmt="none", capsize=2, alpha=0.7 )
            ax_global.set_xlim(FREQ_MIN_HZ, FREQ_MAX_HZ)
            ax_global.set_ylim(POWER_MIN, POWER_MAX)
            ax_global.set_yscale("log")
            ax_global.set_xlabel("Frequency (Hz)")
            ax_global.set_ylabel("Power")
            ax_global.grid(True)

            worker_plots = {}
            for column, w_id in enumerate(sorted(worker_states)):
                # Setup individual worker spectrum plot
                spectrum_axis = fig.add_subplot(grid[1, column])
                spectrum_line, = spectrum_axis.plot([], [], lw=1.5)
                spectrum_axis.set_xlim(FREQ_MIN_HZ, FREQ_MAX_HZ)
                spectrum_axis.set_ylim(POWER_MIN, POWER_MAX)
                spectrum_axis.set_yscale("log")
                spectrum_axis.grid(True)
                spectrum_axis.set_xlabel("Frequency (Hz)")
                spectrum_axis.set_ylabel("Power")

                # Setup text box for worker logs
                log_axis = fig.add_subplot(grid[2, column])
                log_axis.axis("off")
                log_text = log_axis.text(0, 1, "", transform=log_axis.transAxes, va="top", fontsize=8)

                worker_plots[w_id] = {
                    "spectrum_axis": spectrum_axis,
                    "spectrum_line": spectrum_line,
                    "log_text": log_text
                }

        # Draw the latest data on the global plot
        if global_power_means is not None:
            global_power_safe = np.maximum(global_power_means, 1e-12)

            global_line.set_data(global_frequencies, global_power_safe)

            # M2 -> standard deviation
            if global_n_averaged_scans > 1:
                variance = global_power_M2s / (global_n_averaged_scans - 1)

                sigma = np.sqrt(variance)
                global_error.remove()
                global_error = ax_global.errorbar(global_frequencies, global_power_safe, yerr=sigma, fmt="none", capsize=2, alpha=0.7)

            elapsed_time = time.time() - plot_start_time

            ax_global.set_title(f"Cumulative Mean Spectrum ({global_n_averaged_scans} scans, elapsed {elapsed_time:.1f} s)")

        current_time = time.time()
        
        # Iterate over all workers to update their subplots and text logs
        for w_id in sorted(worker_states):
            worker = worker_states[w_id]
            plot = worker_plots[w_id]

            #plot["spectrum_line"].set_data(worker["frequencies"], worker["power_means"])
            plot["spectrum_axis"].set_title(f"Worker {w_id}")

            freqs = worker["frequencies"]
            power = worker["power_means"]
            power = np.maximum(power, 1e-12)

            freqs = worker["frequencies"]
            power = np.maximum(worker["power_means"], 1e-12)
            plot["spectrum_line"].set_data(freqs, power)

            age_seconds = current_time - worker["receive_tss"][-1]
            
            log_lines = [
                f"Total scans: {worker['n_averaged_scans']}",
                f"Age: {age_seconds:.1f} s",
            ]

            # Calculate average time between received updates
            if len(worker["receive_tss"]) < 2:
                log_lines.append("Avg update: --")
            else:
                log_lines.append(f"Avg update: {np.mean(np.diff(worker['receive_tss'])):.2f} s")

            log_lines.extend([
                "",
                "Recent updates (Net & FFT Latency)",
                "-" * 38
            ])
            log_lines.extend(worker["log"])

            # Refresh the text area
            plot["log_text"].set_text("\n".join(log_lines))

        # Render the updated graphics to the screen
        fig.canvas.draw_idle()
        fig.canvas.flush_events()
except eof_exit:
    pass
finally:
    finish_ts = time.time()

    benchmark_output = {
            "throughput": throughput,
            "n_scans_per_batch": n_scans_per_batch,
            "finish_ts": finish_ts,
            "analysis_time": elapsed_time,
            "data": benchmark_records
    }

    with open(BENCHMARK_FILE, "w") as f:
        json.dump(benchmark_output, f, indent=2)

    print(f"Benchmark written to {BENCHMARK_FILE}")

    # Close the Kafka connection
    consumer.close()

input("Press Enter to exit...")

## Benchmark

dire dei vari problemi avuti (la roba di C per checksum in particolare...) e mostrare lscpu dei worker 